# Data Analysis on 23 years of raw climate data

In [ ]:
%load_ext autoreload
%autoreload 2

import sys
sys.path.append('../src')

import os 
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np 

from loaders import *

In [ ]:
df = load_range(2000,1,2023,2)
df.to_parquet("../data/station_data.parquet")

df = pd.read_parquet("../data/station_data.parquet")

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df = df.set_index("date").sort_index()

print(df.index)  # espero un DateTime index 
df.head()

# Data quality graphs

In [ ]:
fig = plt.figure(figsize = (10,5), dpi = 200)

montly_count = df.resample("ME").count()
expected = montly_count.index.days_in_month * 144

missing = montly_count.rsub(expected, axis = 0) # reverse substraction means expected-count

plt.grid()
plt.plot(missing.index, missing["temp"])
plt.xlabel("Year")
plt.ylabel("Number of missing values ")
plt.title("Missing values each year")

In [ ]:
fig = plt.figure(figsize=(10,5), dpi=200)

monthly = df["temp"].resample("ME").mean() # this means resample month ending ME

plt.plot(monthly.index, monthly.values, color="red") # i established that the date is the index
plt.ylabel("Temperature ºC")
plt.xlabel("Year")
plt.title("Temperature 2000-2003")

plt.grid()
plt.show()

In [ ]:
fig = plt.figure(figsize=(10,5), dpi=200)

monthly = df["humidity"].resample("ME").mean() # this means resample month ending ME

plt.plot(monthly.index, monthly.values, color="blue") # i established that the date is the index
plt.ylabel("Humidity %")
plt.xlabel("Year")
plt.title("Humidity 2000-2003")

plt.grid()
plt.show()

In [ ]:
fig = plt.figure(figsize=(10,5), dpi=200)

monthly = df["vapor_pressure"].resample("ME").mean() # this means resample month ending ME

plt.plot(monthly.index, monthly.values, color="skyblue") # i established that the date is the index
plt.ylabel("Vapor pressure")
plt.xlabel("Year")
plt.title("Vapor Pressure 2000-2003")

plt.grid()
plt.show()

In [ ]:
# 1. Ubicar exactamente cuándo ocurre el pico dentro de 2012
subset_2012 = df.loc["2012"]
print(subset_2012["vapor_pressure"].describe())

# Ver los valores más bajos y quién/cuándo son
print(subset_2012.nsmallest(10, "vapor_pressure")[["vapor_pressure"]])

In [ ]:
# ¿Cuánto dura el tramo de ceros el 20 de enero de 2012?
day = df.loc["2012-01-20"]
print(day[["vapor_pressure", "temp", "humidity"]])

In [ ]:
raw = pd.read_excel("../data/A-2012/F2012-01.xls")
raw_jan20 = raw[(raw["Día"] == 20)]
print(raw_jan20[["Día", "H", "T (°C)", "HR %", "TV (kPa)"]].to_string())

In [ ]:
fig = plt.figure(figsize=(10,5), dpi=200)

monthly = df["vapor_deficit"].resample("ME").mean() # this means resample month ending ME

plt.plot(monthly.index, monthly.values, color="skyblue") # i established that the date is the index
plt.ylabel("Vapor deficit")
plt.xlabel("Year")
plt.title("Vapor Deficit 2012-2003")

plt.grid()
plt.show()

In [ ]:
from pathlib import Path

data_dir = Path("../data")
print("data exists:", data_dir.exists())

for year_folder in sorted(data_dir.glob("A-*")):
    files = list(year_folder.iterdir())
    print(f"{year_folder.name}: {[f.name for f in files]}")

In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data")

# Miramos un archivo F y un archivo Fr, año 2011 vs 2013, en crudo (sin asumir fila de cabecera)
sample_files = [
    data_dir / "A-2011" / "F2011-06.xls",
    data_dir / "A-2011" / "Fr-2011-06.xls",
    data_dir / "A-2013" / "F2013-06.xls",
    data_dir / "A-2013" / "Fr-2013-06.xls",
]

for f in sample_files:
    print(f"\n{'='*60}\n{f.name}\n{'='*60}")
    raw = pd.read_excel(f, header=None, nrows=15)
    print(raw)

In [ ]:
import numpy as np

def saturation_vapor_pressure_kpa(temp_c):
    """Tetens equation, returns saturation vapor pressure in kPa."""
    return 0.6108 * np.exp((17.27 * temp_c) / (temp_c + 237.3))

def actual_vapor_pressure_kpa(temp_c, rh_percent):
    """Actual vapor pressure = saturation VP * relative humidity."""
    return saturation_vapor_pressure_kpa(temp_c) * (rh_percent / 100)

# Fila 1 de F2011-06.xls: T=9.85°C, HR=58.93%
print("2011 esperado (kPa):", actual_vapor_pressure_kpa(9.85, 58.93))
print("2011 valor real en archivo:", 0.714)

# Fila 1 de F2013-06.xls: T=11.39158°C, HR=59.88358%
print("2013 esperado (kPa):", actual_vapor_pressure_kpa(11.39158, 59.88358))
print("2013 valor real en archivo (etiquetado kPa):", 8.016384)
print("2013 valor real / 10 (probando si es hPa mal etiquetado):", 8.016384 / 10)


In [ ]:
import pandas as pd
from pathlib import Path

data_dir = Path("../data")

for month in range(1, 13):
    f = data_dir / "A-2011" / f"F2011-{month:02d}.xls"
    if f.exists():
        header_row = pd.read_excel(f, header=None, nrows=1)
        col7_label = header_row.iloc[0, 7]
        print(f"{f.name}: col[7] = {col7_label!r}")
    else:
        print(f"{f.name}: NOT FOUND")